In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("pratt3000/vctk-corpus")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/vctk-corpus


Now we clean the data (note this takes like an hour if you want to use just a subset of the data cut it off early):

In [ ]:
!pip install pyannote.audio
!pip install torch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#this function is used so that no matter where the dataset is stored we can find the data
import os
def find_folder(root_path, target_folder_name):
  for root, dirs, files in os.walk(root_path):
      for dir_name in dirs:
        if dir_name == target_folder_name:
          return os.path.join(root, dir_name)  # Return the full path
  return None  # Folder not found

In [ ]:
#iterate through the dataset and create new folders for speakers numbered 1 to N
original_path = find_folder(path, "wav48")


#get the path of the parent folder to wav48
parent_path = os.path.dirname(original_path)

#make variable for the new path
new_path = os.path.join('/kaggle/working', "cleaned_wav48")

#delete any old cleaned dataset
!rm -rf {new_path}

In [ ]:
#function to clean audio and save to new path

def clean_audio(original_path, new_path, sample_length_sec, pipline):
    #if the audio file does not have a valid audio file header skip it
    try:
      y, sr = librosa.load(original_path)
    except:
      print(f"Rejected audio file at path {original_path} for having invalid audio header")
      return None

    # get the start and end of the voice section of the audio
    vad = pipeline(original_path)
    voice_start = 0
    voice_end = 0

    for speech in vad.get_timeline().support():
      voice_start = speech.start
      voice_end = speech.end
      break

    #skip the audio clip if it has less than sample_length_sec seconds of voice
    if voice_end - voice_start < sample_length_sec:
      # print(f"Rejected audio file at path {original_path} not meeting minimum length")
      return None

    #trim all clips to be 2 second long (note that a sample rate is samples per second)
    start_sample = int(voice_start*sr)
    end_sample = int((voice_start+sample_length_sec)*sr)

    y_processed = y[start_sample:end_sample]

    #if it does not have exaclty sample_length_sec*sr samples skip it (i dont feel like debuging)
    if len(y_processed) != sample_length_sec*sr:
      print(f"Rejected audio file at path {original_path} not meeting minimum length")
      return None

    # print(len(y_processed))
    #apply a hamming window to smooth out the edges of the spliced clip
    window = signal.hamming(len(y_processed))
    y_processed = y_processed*window

    #normalize the signal
    #get the absolute value of the signal
    y_abs = np.abs(y_processed)
    #get max value
    signal_max = np.max(y_abs)
    y_processed = y_processed/signal_max

    #save the new audio file
    sf.write(new_path, y_processed, sr)
    return None


In [ ]:
import os
import re
import librosa
import numpy as np
import soundfile as sf
from pyannote.audio import Pipeline
from scipy import signal

#create the new folder if it does not already exist
if not os.path.exists(new_path):
  os.makedirs(new_path)



#because of how long this takes I added this parameter to control how many speakers we process
#there are 100 speakers in the dataset
max_num_speakers = 2

speaker_index = 0;

#we are going to use the pynote library to detect when the voice is present in the clip
#note this uses a NN model to do so so we will probably have to cite that we used it

file_path = "/content/drive/My Drive/deepL_assignments/huggface_key.txt"
# file_path = "/content/drive/My Drive/SYSEN 5888 Group Project/Project Code/nick_huggface_key.txt"
with open(file_path, 'r') as f:
    huggingface_key = f.read().strip()

#length of each audio sample
sample_length_sec = 2
pipeline = Pipeline.from_pretrained("pyannote/voice-activity-detection", use_auth_token=huggingface_key)

#iterate through each speakers utterances in order
for folder_name in sorted(os.listdir(original_path), key=lambda x: int(re.search(r'\d+', x).group())):
  #get the full apth of the original speaker folders
  folder_path = os.path.join(original_path, folder_name)

  #make new folder in cleaned folder to for each speaker with a new name and number
  speaker_folder_name = f"speaker_{speaker_index}_audio_files"
  speaker_folder_path = os.path.join(new_path, speaker_folder_name)

  #create the new speaker folder if it does not already exist
  if not os.path.exists(speaker_folder_path):
    os.makedirs(speaker_folder_path)

  #utterance counter
  utterance_index = 0

  #iterate through each speakers utterance
  for audio_file_path in os.listdir(folder_path):
    original_audio_file_path = os.path.join(original_path, folder_name, audio_file_path)
    #save the edited clip to the new folder
    utterance_name = f"speaker_{speaker_index}_utterance_{utterance_index}.wav"
    utterance_path = os.path.join(speaker_folder_path, utterance_name)
    clean_audio(original_audio_file_path, utterance_path, sample_length_sec, pipeline)
    utterance_index += 1

  print(f"Finished speaker {speaker_index}")
  speaker_index += 1

  if speaker_index >= max_num_speakers:
    break

print("Created edited dataset")


config.yaml:   0%|          | 0.00/277 [00:00<?, ?B/s]

DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _speechbrain_save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _speechbrain_load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _recover


pytorch_model.bin:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

config.yaml:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/pytorch_lightning/utilities/migration/migration.py:208: You have multiple `ModelCheckpoint` callback states in this checkpoint, but we found state keys that would end up colliding with each other after an upgrade, which means we can't differentiate which of your checkpoint callbacks needs which states. At least one of your `ModelCheckpoint` callbacks will not be able to reload the state.
INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.1.3 to v2.5.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/torch/pyannote/models--pyannote--segmentation/snapshots/059e96f964841d40f1a5e755bb7223f76666bba4/pytorch_model.bin`


Model was trained with pyannote.audio 0.0.1, yours is 3.3.2. Bad things might happen unless you revert pyannote.audio to 0.x.
Model was trained with torch 1.7.1, yours is 2.6.0+cu124. Bad things might happen unless you revert torch to 1.x.
Finished speaker 0
Finished speaker 1
Created edited dataset


Import the libri speech dataset and clean it (not done skip this for now)



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yasiashpot/librispeech")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/librispeech


In [ ]:
original_libri_path = find_folder(path, "dev-clean")

with open(file_path, 'r') as f:
    huggingface_key = f.read().strip()

# note this dataset has 47 speakers in it
speaker_index=99
libri_max_num_speakers = 0

#length of each audio sample
sample_length_sec = 2
pipeline = Pipeline.from_pretrained("pyannote/voice-activity-detection", use_auth_token=huggingface_key)

#iterate through each speakers utterances in order
for folder_name in sorted(os.listdir(original_libri_path)):
  #get the full apth of the original speaker folders
  folder_path = os.path.join(original_libri_path, folder_name)

  #utterance counter
  utterance_index = 0

  #make new folder in cleaned folder to for each speaker with a new name and number
  speaker_folder_name = f"speaker_{speaker_index}_audio_files"
  speaker_folder_path = os.path.join(new_path, speaker_folder_name)

  #create the new speaker folder if it does not already exist
  if not os.path.exists(speaker_folder_path):
    os.makedirs(speaker_folder_path)

  #get folder path of each of the speakers data folders and iterate
  for subfolder_name in os.listdir(folder_path):
    subfolder_path = os.path.join(folder_path, subfolder_name)

    #iterate through all of the audio files in the folder
    for audio_file_path in os.listdir(subfolder_path):
      original_audio_file_path = os.path.join(subfolder_path, audio_file_path)
      utterance_name = f"speaker_{speaker_index}_utterance_{utterance_index}.wav"
      utterance_path = os.path.join(speaker_folder_path, utterance_name)
      clean_audio(original_audio_file_path, utterance_path, sample_length_sec, pipeline)
      utterance_index += 1

  print(f"Finished speaker {speaker_index}")
  speaker_index += 1

  if speaker_index >= libri_max_num_speakers:
    break

print("Added Libri to the Dataset")

/usr/local/lib/python3.11/dist-packages/pytorch_lightning/utilities/migration/migration.py:208: You have multiple `ModelCheckpoint` callback states in this checkpoint, but we found state keys that would end up colliding with each other after an upgrade, which means we can't differentiate which of your checkpoint callbacks needs which states. At least one of your `ModelCheckpoint` callbacks will not be able to reload the state.
INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.1.3 to v2.5.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/torch/pyannote/models--pyannote--segmentation/snapshots/059e96f964841d40f1a5e755bb7223f76666bba4/pytorch_model.bin`


Model was trained with pyannote.audio 0.0.1, yours is 3.3.2. Bad things might happen unless you revert pyannote.audio to 0.x.
Model was trained with torch 1.7.1, yours is 2.6.0+cu124. Bad things might happen unless you revert torch to 1.x.


<ipython-input-6-e8dfe26babe2>:6: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(original_path)
/usr/local/lib/python3.11/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Rejected audio file at path /kaggle/input/librispeech/LibriSpeech/dev-clean/1272/135031/1272-135031.trans.txt for having invalid audio header


<ipython-input-6-e8dfe26babe2>:6: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(original_path)
/usr/local/lib/python3.11/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Rejected audio file at path /kaggle/input/librispeech/LibriSpeech/dev-clean/1272/128104/1272-128104.trans.txt for having invalid audio header


<ipython-input-6-e8dfe26babe2>:6: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(original_path)
/usr/local/lib/python3.11/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Rejected audio file at path /kaggle/input/librispeech/LibriSpeech/dev-clean/1272/141231/1272-141231.trans.txt for having invalid audio header
Finished speaker 99
Added Libri to the Dataset


Feature Extraction

In [ ]:
!pip install librosa==0.11.0
!pip install numpy==2.0.0
#!pip install --upgrade numpy
import librosa
import numpy as np

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 74.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2


In [ ]:
def extract_mel_spectrogram(audio_file_path):
    NUM_MELS = 40
    #length of window in seconds
    WINDOW_LEN = 25/1000 #25 miliseconds
    STEP_TIME =  10/1000 #10 miliseconds
    y, sr = librosa.load(audio_file_path)

    win_sample_len= int(WINDOW_LEN*sr)
    step_sample_len = int(STEP_TIME*sr)

    spectrogram = librosa.feature.melspectrogram(y=y,
                                                 sr=sr,
                                                 n_mels=NUM_MELS,
                                                 hop_length=step_sample_len,
                                                 win_length=win_sample_len
                                                 )
    log_S = librosa.power_to_db(spectrogram, ref=np.max)
    return log_S

Iterated through the cleaned dataset and construct vectors to be used in training. Each example will contain the mel spectrum and the speaker's label


In [ ]:
import os

data_set_path = new_path
speaker_data = []
speaker_labels = []

#control the number of data points we will create. purley for the sake of time
#set this equal to "all" if you want to use the whole dataset
num_data_points = "all"
count = 0
for filename in os.listdir(data_set_path):
        if num_data_points != "all" and count > num_data_points:
          break
        #go through all the speakers' folder
        speaker_folder_path = os.path.join(data_set_path, filename)

        if os.path.isdir(speaker_folder_path):
            for audio_file in os.listdir(speaker_folder_path):
                y, sr = librosa.load(os.path.join(speaker_folder_path, audio_file))
                mel = extract_mel_spectrogram(os.path.join(speaker_folder_path, audio_file))
                speaker_data.append(mel)

                #extract the speaker number from the file name
                speaker_number = int(filename.split("_")[1])
                speaker_labels.append(speaker_number)

                count += 1

                if num_data_points != "all" and count > num_data_points:
                  break
speaker_data = np.stack(speaker_data, axis=0)
speaker_labels = np.array(speaker_labels)


Now we need to construct a dataset with custom batches. From looking at the dataset each speaker appears to have about 100 utterances so we will make each batch have 10 utterances from each speaker.

In [ ]:
import tensorflow as tf

class SpeakerBatchGenerator(tf.keras.utils.Sequence):
  def __init__(self, data, labels, speaker_examples_per_batch = 10, speakers_per_batch = 20):
    super(SpeakerBatchGenerator, self).__init__()
    self.data = data
    self.labels = labels
    self.possible_speaker_labels = np.unique(labels)
    self.speaker_examples_per_batch = speaker_examples_per_batch
    self.speakers_per_batch = speakers_per_batch
    self.data_per_batch = self.speaker_examples_per_batch*self.speakers_per_batch

    print(f"There are {len(self.possible_speaker_labels)} possible different speakers to be used for training")

    print("You have chosen:")
    print(f"Each batch will have {self.speakers_per_batch} different speakers")
    print(f"Each speaker will have {self.speaker_examples_per_batch} examples")
    print(f"There are {self.data_per_batch} data points per batch")


    #figure out what examples correspond to what labels
    self.speaker_indecies = {}
    for speaker_label in self.possible_speaker_labels:
      self.speaker_indecies[speaker_label] = np.where(labels == speaker_label)[0]

    #create batches and save their indecies in the dataset
    self.batches = self.create_batches()

  #this function goes through the dataset an creates a list containing arrays with the indeces
  #for each batch
  @tf.function
  def create_batches(self):
    batches = []
    #construct each batch
    for batch in range(self.__len__()):
      batch_indecies = []

      #Break if there is no more data
      if sum([len(self.speaker_indecies[speaker_label]) for speaker_label in self.possible_speaker_labels]) == 0:
        break
      #keep track of how many speakers we have put in the batch
      batch_speaker_count = 0

      for speaker_label in self.possible_speaker_labels:
        if batch_speaker_count >= self.speakers_per_batch:
          break
        #Break if there is no more data
        if sum([len(self.speaker_indecies[speaker_label]) for speaker_label in self.possible_speaker_labels]) == 0:
          break

        #skip speaker if there are less than 2 examples (this shouldnt happen though)
        if len(self.speaker_indecies[speaker_label]) < 2:
          continue

        #if there are not enough examples to get self.speaker_examples_per_batch just use whats left
        if len(self.speaker_indecies[speaker_label]) <= self.speaker_examples_per_batch:
          speaker_indeces_for_current_batch = self.speaker_indecies[speaker_label]
          batch_indecies.extend(speaker_indeces_for_current_batch)
        else:

          #choose speaker_examples_per_batch examples without replacement from our list of speaker examples
          speaker_indeces_for_current_batch = np.random.choice(self.speaker_indecies[speaker_label], self.speaker_examples_per_batch, replace=False)

          batch_indecies.extend(speaker_indeces_for_current_batch)

          #remove the chosen indecies from the list for the speaker since we dont want to use them twice
          removal_mask = ~np.isin(self.speaker_indecies[speaker_label], speaker_indeces_for_current_batch)
        batch_speaker_count += 1
      batches.append(batch_indecies)

    return batches

  #given the index for the specifed batch, get its data and labels
  @tf.function
  def get_batch_data(self, index):
    batch_indecies = self.batches[index]
    batch_data = self.data[batch_indecies]
    batch_labels = self.labels[batch_indecies]

    # print("labels for batch")
    # for i in batch_labels:
    #   tf.print(i)
    # print("end of listing")

    return batch_data, batch_labels

  @tf.function
  def __len__(self):
    return self.data.shape[0] // self.data_per_batch
  @tf.function
  def __getitem__(self, index):
    return self.get_batch_data(index)



Code to confirm the matrix operations are correct for the loss function


In [ ]:
import numpy as np
import tensorflow as tf

#confirm positive calculations

#dummy embeddings
embeddings_d = np.array([[1, 2, 3], [1, 4, 7], [-4, -5, -6], [-6, -1, -1]], dtype=np.float32)
embeddings_d = tf.convert_to_tensor(embeddings_d)
embeddings_d = tf.math.l2_normalize(embeddings_d, axis=1)

#dummy label vector
labels_d = np.array([0, 0, 4, 4], dtype=np.int32)
labels_d = tf.convert_to_tensor(labels_d)

unique_labels, new_labels = tf.unique(labels_d)

# print(unique_labels)

num_speakers = tf.shape(unique_labels)[0]

# print(num_speakers)

centroids = tf.math.unsorted_segment_mean(embeddings_d, new_labels, num_segments=num_speakers)

# print(centroids)

corresponding_centroids = tf.gather(centroids, new_labels)

# print("corresponding centroids")
# print(corresponding_centroids)

embedding_speaker_labels = tf.gather(unique_labels, new_labels)

# print("embedding speaker labels")
# print(embedding_speaker_labels)

speaker_embedding_count = tf.math.bincount(new_labels)

# print("speaker embedding count")
# print(speaker_embedding_count)

embeddings_with_same_speaker_count = tf.gather(speaker_embedding_count, new_labels)

# print("embeddings with same speaker count")
# print(embeddings_with_same_speaker_count)

embeddings_with_same_speaker_i_removed_count = embeddings_with_same_speaker_count - 1

# print("embeddings with same speaker i removed count")
# print(embeddings_with_same_speaker_i_removed_count)

embeddings_with_same_speaker_i_removed_count_inverse = tf.cast(1/embeddings_with_same_speaker_i_removed_count, dtype=tf.float32)

embeddings_with_same_speaker_i_removed_count_inverse_reshaped = tf.expand_dims(embeddings_with_same_speaker_i_removed_count_inverse, axis=-1)

# print("embeddings with same speaker i removed count inverse")
# print(embeddings_with_same_speaker_i_removed_count_inverse)

# print("corresponding centroids")
# print(corresponding_centroids)

#calculate the i removed centroid
centroids_i_removed = corresponding_centroids*embeddings_with_same_speaker_i_removed_count_inverse_reshaped*tf.reshape(tf.cast(embeddings_with_same_speaker_count, dtype=tf.float32), (-1, 1))

# print("centroids i removed")
# print(centroids_i_removed)

centroids_i_removed -= embeddings_d*embeddings_with_same_speaker_i_removed_count_inverse_reshaped

# print("centroids i removed actual")
# print(centroids_i_removed)

similarity = tf.reduce_sum(embeddings_d * centroids_i_removed, axis=1)

# print("similarity")
# print(similarity)

#confirm negative calculations

batch_size = tf.shape(embeddings_d)[0]

# print("batch size")
# print(batch_size)

expanded = tf.expand_dims(centroids, axis=0)
tiled = tf.tile(expanded, [batch_size, 1, 1])

# print("tiled")
# print(tiled)

one_hot_matrix = tf.one_hot(new_labels, depth=num_speakers)

# print("one hot matrix")
# print(one_hot_matrix)

one_hot_expanded = tf.expand_dims(one_hot_matrix, axis=-1)      # (B, num_speakers, 1)
centroids_expanded = tf.expand_dims(corresponding_centroids, axis=1)  # (B, 1, E)
# print("one hot expanded")
# print(one_hot_expanded)
# print("centroids expanded")
# print(centroids_expanded)
for_old_centroid_removal_matrix = one_hot_expanded * centroids_expanded  # (B, num_speakers, E)

# print("for old centroid removal matrix")
# print(for_old_centroid_removal_matrix)

# print(centroids_i_removed)

centroids_i_removed_expanded = tf.expand_dims(centroids_i_removed, axis=1)
for_old_centroid_removal_matrix_i_removed = one_hot_expanded * centroids_i_removed_expanded  # (B, num_speakers, E)

# print("for old centroid removal matrix i removed")
# print(for_old_centroid_removal_matrix_i_removed)

new_centroids = tiled - for_old_centroid_removal_matrix + for_old_centroid_removal_matrix_i_removed

# print("new centroids")
# print(new_centroids)

embeddings_expanded = tf.expand_dims(embeddings_d, axis=1)

dot_product_matrix = tf.reduce_sum(embeddings_expanded * new_centroids, axis=2)

print("dot product matrix")
print(dot_product_matrix)





dot product matrix
tf.Tensor(
[[ 0.9869275  -0.7257711 ]
 [ 0.9869275  -0.6326387 ]
 [-0.95022595  0.6470396 ]
 [-0.40818387  0.6470396 ]], shape=(4, 2), dtype=float32)


Construct the model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Lambda, Input
import tensorflow as tf
from tensorflow.keras import layers, saving
import keras
from tensorflow.keras.initializers import HeNormal
from tensorflow.keras.regularizers import l2

@tf.keras.utils.register_keras_serializable()
class LSTM_Embedding_Model(tf.keras.Model):
  def __init__(self, init_b = -0.2, init_w = 0.45, input_shape = (40, 201), **kwargs):
    super(LSTM_Embedding_Model, self).__init__(**kwargs)
    self.loss_tracker = tf.keras.metrics.Mean(name="loss")
    self.input_shape = input_shape

    #create the trainable w value for the similariy scale calculation
    self.W = self.add_weight(
                              shape=(),
                              initializer=tf.keras.initializers.Constant(init_w),
                              trainable=True,
                              name="GE2E_W"
                            )
    #create the trainable b value for the similariy bias calculation
    self.b = self.add_weight(
                              shape=(),
                              initializer=tf.keras.initializers.Constant(init_b),
                              trainable=True,
                              name="GE2E_b"
                            )

    self.model = self.build_model()


  @property
  def metrics(self):
      # Return a list of metrics to reset/track each epoch
      return [self.loss_tracker]

  def build_model(self):
    model = Sequential()
    model.add(Input(shape=self.input_shape))
    # First LSTM layer
    model.add(LSTM(units=768, return_sequences=True, input_shape=self.input_shape))
    # Second LSTM layer
    model.add(LSTM(units=768, return_sequences=True))
    # Third LSTM layer
    model.add(LSTM(units=768, return_sequences=False))
    # Dense layer
    model.add(Dense(units=256, activation='relu', kernel_initializer=HeNormal()))
    return model

  def call(self, input, training=False):
    return self.model(input, training=training)

  def get_config(self):
    config = super(LSTM_Embedding_Model, self).get_config()
    config.update({
        'init_b': self.b.numpy(),  # Include the value of b
        'init_w': self.W.numpy(),  # Include the value of W
        'input_shape': self.input_shape,
    })
    return config

  @classmethod
  def from_config(cls, config):
    return cls(**config)

  #custom training loop to calculate GE2E loss
  #loss function based on this paper https://arxiv.org/pdf/1710.10467
  @tf.function
  def train_step(self, data):

    batch_data, batch_labels = data

    with tf.GradientTape() as tape:
      loss = tf.constant(0.0)

      #calculate the embeddings for each data point
      embeddings = self.model(batch_data, training=True)
      #normalize embeddings
      embeddings = tf.math.l2_normalize(embeddings, axis=1)


      # print(embeddings)

      #get info on the speakers we have
      #unique labels are the labels of all speakers present
      #new_labels are the mappings of the labels in batch_labels to the unique_labels vector
      unique_labels, new_labels = tf.unique(batch_labels)

      #number of speakers
      num_speakers = tf.shape(unique_labels)[0]

      #-----------------------------------------------------#
      #calculate the "positive" loss component look at the equation 6 in the error paper

      #calculate the centroids
      centroids = tf.math.unsorted_segment_mean(embeddings, new_labels, num_segments=num_speakers)

      # print("embeddings are:")
      # print(embeddings)
      # print("centroids are:")
      # print(centroids)

      #matrix where each row is the centroid corresponding to the speaker of the embedding
      #it lines up with
      corresponding_centroids = tf.gather(centroids, new_labels)

      #in order to avoid trivial solutions we must remove the embedding of interest from the centroid's calculation
      #see equation 8

      #calculate the i excluded centroid vector:

      #get the speaker for each embedding in the batch as a vector
      embedding_speaker_labels = tf.gather(unique_labels, new_labels)

      #create a vector where each row is the count of each speakers embeddings
      #the vector should look like this [speaker_0_count, speaker_1_count ... speaker_N_count]
      speaker_embedding_count = tf.math.bincount(new_labels)

      #take embedding_speaker_labels and make vector with rows replaced with the speaker counts
      #this vector should be the corresponding speaker_embedding_count to every embeddings's speaker
      #in embedding_speaker_labels
      embeddings_with_same_speaker_count = tf.gather(speaker_embedding_count, new_labels)

      #since we will be removing an embedding from the count subtract 1 from each count
      embeddings_with_same_speaker_i_removed_count = embeddings_with_same_speaker_count - 1

      #take the recirpocal of every count
      embeddings_with_same_speaker_i_removed_count_inverse = tf.cast(1/embeddings_with_same_speaker_i_removed_count, dtype=tf.float32)

      embeddings_with_same_speaker_i_removed_count_inverse_reshaped = tf.expand_dims(embeddings_with_same_speaker_i_removed_count_inverse, axis=-1)

      #calculate the i removed centroid
      centroids_i_removed = corresponding_centroids*embeddings_with_same_speaker_i_removed_count_inverse_reshaped*tf.reshape(tf.cast(embeddings_with_same_speaker_count, dtype=tf.float32), (-1, 1))
      centroids_i_removed -= embeddings*embeddings_with_same_speaker_i_removed_count_inverse_reshaped

      #compute the cosine similarity between each embedding and its speaker's i removed centroid
      # Assuming embeddings has shape (B, E) and centroids_i_removed has shape (B, E)
      pos_dot_product = tf.reduce_sum(embeddings * centroids_i_removed, axis=1)

      # print("/n")
      # print("positive similarity")
      # print(tf.reduce_mean(similarity))

      #apply the learnable parameters
      similarity = self.W*pos_dot_product + self.b



      positive_loss = -1*similarity

      #-----------------------------------------------------#
      #calculate the "negative" loss component look at the equation 6 in the error paper

      #we must fist calculate a 3d matrix of shape B x N x E where all the rows
      #are identical and are the centroid vectors for all the speakers

      #it should look like this where each cx is a vector

      """
      |c1|c2|...|cN|
      |c1|c2|...|cN|
      |c1|c2|...|cN|
      |c1|c2|...|cN|
      ...
      |c1|c2|...|cN|

      B rows and N columns
      """
      batch_size = tf.shape(embeddings)[0]
      expanded = tf.expand_dims(centroids, axis=0)
      tiled = tf.tile(expanded, [batch_size, 1, 1])

      #to edit the previous matrix to account for the i removed centroids we need two
      #matrices we need a matrix where all elements are zero except centroids which we must
      #subtract to make room for the new i removed centroid. it looks like this. It should be of shape

      """
      |c1|0|...|0|
      |c1|0|...|0|
      ...
      |c1|0|...|0|
      |0|c2|...|0|
      |0|c2|...|0|
      ...
      |0|c2|...|0|
      ...
      |0|...|0|cN|
      |0|...|0|cN|
      ...
      |0|...|0|cN|

      B rows and N columns
      """

      one_hot_matrix = tf.one_hot(new_labels, depth=num_speakers)
      one_hot_expanded = tf.expand_dims(one_hot_matrix, axis=-1)      # (B, num_speakers, 1)
      centroids_expanded = tf.expand_dims(corresponding_centroids, axis=1)  # (B, 1, E)
      for_old_centroid_removal_matrix = one_hot_expanded * centroids_expanded  # (B, num_speakers, E)

      #Now we must make a matrix which is  exactly the same but the centroid are i removed

      """
      |c1(-1)|0|...|0|
      |c1(-2)|0|...|0|
      ...
      |c1(-M)|0|...|0|
      |0|c2(-1)|...|0|
      |0|c2(-2)|...|0|
      ...
      |0|c2(-M)|...|0|
      ...
      |0|...|0|cN(-1)|
      |0|...|0|cN(-2)|
      ...
      |0|...|0|cN(-M)|

      B rows and N columns
      """

      centroids_i_removed_expanded = tf.expand_dims(centroids_i_removed, axis=1)
      for_old_centroid_removal_matrix_i_removed = one_hot_expanded * centroids_i_removed_expanded  # (B, num_speakers, E)

      #get new centroid matrix for calculation
      new_centroids = tiled - for_old_centroid_removal_matrix + for_old_centroid_removal_matrix_i_removed


      embeddings_expanded = tf.expand_dims(embeddings, axis=1)

      dot_product_matrix = tf.reduce_sum(embeddings_expanded * new_centroids, axis=2)

      print("loss gap")
      print(tf.reduce_mean(pos_dot_product)-tf.reduce_mean(dot_product_matrix))

      print("dot product matrix")
      print(tf.reduce_mean(dot_product_matrix))


      similarity_matrix = self.W*dot_product_matrix + self.b

      # take the exponential of every element in the matrix
      exp_similarity_matrix = tf.math.exp(similarity_matrix)

      #sum across the rows
      exp_similarity_matrix_sum = tf.reduce_sum(exp_similarity_matrix, axis=1)




      #take the log of the sum
      negative_loss = tf.math.log(exp_similarity_matrix_sum)

      #sum to get batch loss
      loss = tf.reduce_mean(negative_loss + positive_loss)

    gradients = tape.gradient(loss, self.trainable_variables)
    # print(gradients)
    self.optimizer.apply_gradients(zip(gradients, self.trainable_variables))
    self.loss_tracker.update_state(loss)
    return {"loss": self.loss_tracker.result()}


If it complains about the memory being full run this rather than resatrting

In [ ]:
# ipython-input-118-ba8634324ea8
import tensorflow as tf
# Clear the session
tf.keras.backend.clear_session()
# Reset the default graph
tf.compat.v1.reset_default_graph()
# Immediately release memory
# tf.compat.v1.disable_eager_execution() # Comment out this line to enable eager execution
tf.compat.v1.enable_eager_execution()
with tf.device('/gpu:0'): # Replace with your desired GPU device
    tf.compat.v1.get_default_graph() # Recreates a graph on the specified device

In [ ]:
training_data = SpeakerBatchGenerator(
                                      speaker_data,
                                      speaker_labels,
                                      speaker_examples_per_batch=10,
                                      speakers_per_batch=2
                                     )

There are 3 possible different speakers to be used for training
You have chosen:
Each batch will have 2 different speakers
Each speaker will have 10 examples
There are 20 data points per batch


In [ ]:
model = LSTM_Embedding_Model(init_b = -0.2, init_w = 0.45, input_shape = speaker_data[0].shape)
model.summary()


Model: "lstm__embedding__model_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ sequential_1 (Sequential)            │ (None, 256)                 │      12,620,032 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 12,620,034 (48.14 MB)

 Trainable params: 12,620,034 (48.14 MB)

 Non-trainable params: 0 (0.00 B)

Start training

In [ ]:
tf.config.run_functions_eagerly(True)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002))
history = model.fit(training_data, epochs=100, verbose = 1)

Epoch 1/100


/usr/local/lib/python3.11/dist-packages/tensorflow/python/data/ops/structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


loss gap
tf.Tensor(0.005009532, shape=(), dtype=float32)
dot product matrix
tf.Tensor(0.7566495, shape=(), dtype=float32)
 1/21 ━━━━━━━━━━━━━━━━━━━━ 1:11 4s/step - loss: 0.6909loss gap
tf.Tensor(0.006669998, shape=(), dtype=float32)
dot product matrix
tf.Tensor(0.8261391, shape=(), dtype=float32)
 2/21 ━━━━━━━━━━━━━━━━━━━━ 17s 896ms/step - loss: 0.6907loss gap
tf.Tensor(-0.004373908, shape=(), dtype=float32)
dot product matrix
tf.Tensor(0.8550819, shape=(), dtype=float32)
 3/21 ━━━━━━━━━━━━━━━━━━━━ 15s 886ms/step - loss: 0.6912loss gap
tf.Tensor(0.004196167, shape=(), dtype=float32)
dot product matrix
tf.Tensor(0.8898279, shape=(), dtype=float32)
 4/21 ━━━━━━━━━━━━━━━━━━━━ 14s 878ms/step - loss: 0.6914loss gap
tf.Tensor(0.0075615644, shape=(), dtype=float32)
dot product matrix
tf.Tensor(0.8972093, shape=(), dtype=float32)
 5/21 ━━━━━━━━━━━━━━━━━━━━ 14s 886ms/step - loss: 0.6914loss gap
tf.Tensor(9.840727e-05, shape=(), dtype=float32)
dot product matrix
tf.Tensor(0.9048177, shape=(), dt

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.


KeyboardInterrupt



In [ ]:
import matplotlib.pyplot as plt
def plot_loss(history):
    plt.plot(history.history['loss'], label='loss')
    #plt.plot(history.history['val_loss'], label='val_loss')
    #plt.ylim([min(history.history['loss']), max(history.history['loss'])])
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()
plot_loss(history)

In [ ]:
model_save_path = '/content/drive/My Drive/deepL_assignments/embedding_model_300_epochs.keras'  # save the model because training this is a bitch
model.save(model_save_path)


In [ ]:
from tensorflow import keras
model_save_path = "/content/drive/My Drive/deepL_assignments/embedding_model_300_epochs.keras"
loaded_model = keras.models.load_model(model_save_path)

Test the model to make sure that the model actually functions as want. I.e that audio clips of the same speaker are similar.


In [ ]:
#note I know I am testing on data already seen I am just trying to see
#if it works at all

num_speakers = 4
test_data = SpeakerBatchGenerator(
                                      speaker_data,
                                      speaker_labels,
                                      speaker_examples_per_batch=4,
                                      speakers_per_batch=num_speakers
                                     )

batch_data, batch_labels = test_data[0]

embeddings = loaded_model.model(batch_data, training=False)



Graph the embeddings

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import umap

# Reduce the dimensionality to 2D using UMAP
umap_reducer = umap.UMAP(n_components=2, random_state=42)
embeddings_2d = umap_reducer.fit_transform(embeddings)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1],
                      c=np.array(batch_labels), cmap='viridis', alpha=0.7)
plt.colorbar(scatter, label='Speaker Label')
plt.title("UMAP Visualization of Embeddings")
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.show()